# EVSI Validation Notebook

A deterministic EVSI example using the public sample-information API.

In [ ]:
import numpy as np
from voiage.analysis import DecisionAnalysis
from voiage.schema import ValueArray, ParameterSet, TrialDesign, DecisionOption
from voiage.methods.sample_information import evsi, enbs

np.random.seed(9)
psa_prior = ParameterSet.from_numpy_or_dict({
    'mean_new_treatment': np.linspace(9.0, 11.0, 80),
    'mean_standard_care': np.linspace(7.5, 9.0, 80),
    'unrelated_param': np.linspace(0.0, 1.0, 80),
})
trial_design = TrialDesign(arms=[
    DecisionOption(name='New Treatment', sample_size=25),
    DecisionOption(name='Standard Care', sample_size=25),
])

def model(psa):
    treatment = psa.parameters['mean_new_treatment']
    standard = psa.parameters['mean_standard_care']
    unrelated = psa.parameters['unrelated_param']
    values = np.column_stack([
        100.0 + 8.0 * standard,
        85.0 + 10.0 * treatment + 2.0 * unrelated,
    ])
    return ValueArray.from_numpy(values, ['standard', 'new'])

evsi_value = evsi(
    model_func=model,
    psa_prior=psa_prior,
    trial_design=trial_design,
    method='efficient',
)
print(f'EVSI: {evsi_value:.4f}')


In [ ]:
enbs_value = enbs(evsi_value, 12.5)
print(f'ENBS (with fixed research cost): {enbs_value:.4f}')
